# Convert a LittleBit checkpoint to a mobile GGUF

This notebook downloads the checkpoint, builds the repository's CPU quantizer, and converts the model to a mobile-friendly GGUF. The Q4_K embeddings and output head keep the resulting file around 1.5 GB (actual size depends on checkpoint details).

**License:** `LittleBitLLM/littlebit-qwen3-8b-0.55bpw` is distributed under CC BY-NC 4.0. Use the checkpoint and converted model for non-commercial purposes only, and comply with the license terms. The notebook downloads weights at run time; this repository commits no model weights.

In [ ]:
MODEL_ID = "LittleBitLLM/littlebit-qwen3-8b-0.55bpw"  # Change to another compatible LittleBit checkpoint if desired.
REPO_DIR = "/content/littlebit-gguf"
CHECKPOINT_DIR = "/content/littlebit-checkpoint"
OUTPUT_FILE = "/content/littlebit-qwen3-8b-q4_k.gguf"

## Prepare the converter and quantizer

Run this cell once in a fresh Colab runtime. It checks out the converter with its patched llama.cpp submodule, installs Python requirements, and builds the CPU tools required for Q4_K quantization.

In [ ]:
!git clone --recurse-submodules https://github.com/GeniusDevam/littlebit-gguf.git "$REPO_DIR"
%cd $REPO_DIR
!python -m pip install -r requirements.txt
!python -m pip install cmake ninja
!bash scripts/build.sh

## Download the checkpoint

The model ID is configurable in the first code cell. The checkpoint files are stored in the Colab runtime, not in this repository.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id=MODEL_ID, local_dir=CHECKPOINT_DIR)
print(f"Downloaded {MODEL_ID} to {CHECKPOINT_DIR}")

## Convert to Q4_K GGUF

The repository converter uses `--embd-type q4_k` for Q4_K embeddings and, by default, applies that same type to the output tensor. `--output-type q4_k` is included explicitly here. Conversion uses the CPU library built above.

In [ ]:
!python convert_littlebit_to_gguf.py "$CHECKPOINT_DIR" --outfile "$OUTPUT_FILE" --embd-type q4_k --output-type q4_k --validate 2

In [ ]:
from pathlib import Path

print(f"Created {OUTPUT_FILE} ({Path(OUTPUT_FILE).stat().st_size / 1e9:.2f} GB)")

## Optional: copy the GGUF to Google Drive

Run the cell below only if you want to keep the converted file after the Colab runtime ends. Mounting Drive grants this notebook access to the selected Drive account.

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
drive_output = "/content/drive/MyDrive/littlebit-qwen3-8b-q4_k.gguf"
shutil.copy2(OUTPUT_FILE, drive_output)
print(f"Copied converted GGUF to {drive_output}")